# 04 · GBIF presence and biotic opportunity

**Data stream:** biotic opportunity (which interaction partners of a species
already occur in a country).
**Output feeds:** the presence matrix and Stream E (biotic opportunity) of
the model: `gbif_presence.pt`.

---

## What this notebook does

For every species in the graph and every SInAS location it builds

* **`presence[s, c]`** — species *s* is recorded in location *c*
  (SInAS record of any status, or GBIF occurrences);
* **`pos_opp_mat[s, c]`** — fraction of *s*'s **positive** partners
  (food, hosts, mutualists, habitat) present in *c*;
* **`neg_opp_mat[s, c]`** — fraction of *s*'s **negative** partners
  (predators, parasites, pathogens, competitors) present in *c*.

Which partner is positive or negative comes from the per-term valences of
notebook 03 (`source_valence` / `target_valence`), so it is defined in one
place only.

**Higher-order interactions (HOI)** are included: a partner known only at
genus or family level (e.g. "eats *Quercus*") counts as present in a
location when any species of that genus / family is present there, and a
record about a genus (e.g. "*Quercus* is eaten by *Lymantria dispar*") is
passed on to the member species of that genus.

## Two ways to get GBIF presence (`PRESENCE_SOURCE`)

| | `country_api` (default) | `gadm_sql` |
|---|---|---|
| How | GBIF occurrence API, one request per species (`facet=country`) | GBIF **SQL downloads**: one job for all species × countries, one for all species × GADM level-1 regions of split countries |
| Reuses the previous fetch | yes; only new species are queried | no (fresh, consistent data); or `GADM_COUNTRY_SOURCE = "existing"` to keep the country data |
| Sub-national locations (Hawaii, Canary Islands, Sicily, …) | get their **whole country's** presence (over-marking) | get only their **own region's** records |
| Record filter | none (as the previous fetch) | `SQL_WHERE`: present, no fossils, no zoo / garden specimens |
| Needs | internet | internet + free GBIF account |
| Time | ~10–25 min per 15,000 species | minutes to a few hours of GBIF processing (unattended) + one download |
| Citable | GBIF.org + access date | a **DOI** per download |

Everything after Step 4b (SInAS presence, partners, opportunity, outputs)
is identical for both; only `presence_gbif` differs.

## What is reused and what is new

The GBIF country counts fetched by the previous notebook
(`fetch_gbif_presabs_country_pos_neg_split.ipynb`, saved in
`gbif_presence.pt`) are **reused**; nothing is re-downloaded for species
that already have data. Only species that are new in the graph (e.g.
SInAS species without GloBI interactions, which notebook 03 now keeps) are
fetched, with the same type of GBIF query.

## Pipeline at a glance

| Step | What happens | Main output |
|---|---|---|
| 0 | Configuration, sources, helper functions | — |
| 1 | Species universe (same definition as the model) and location list | `species_ids`, `locations` |
| 2 | Location → GBIF country codes | `iso2_to_cols` |
| 3 | Reuse the previous GBIF presence, re-keyed to the new species IDs | `presence_gbif` |
| 4 | `country_api`: fetch GBIF presence for species without previous data | `presence_gbif` |
| 4b | `gadm_sql`: GBIF SQL downloads, country + GADM level-1 presence | `presence_gbif` |
| 5 | SInAS presence and combined presence matrix | `presence` |
| 6 | Presence of higher taxa (genus / family) for HOI partners | `presence_higher` |
| 7 | Partner lists with valence (species-level + HOI) | `triplets` |
| 8 | Biotic opportunity matrices | `pos_opp`, `neg_opp` |
| 9 | QA, incl. comparison with the previous matrices | `qa` |
| 10 | Write outputs | files in `data/` |

## Outputs (written to `data/`)

| File | Content |
|---|---|
| `gbif_presence.pt` | Same keys as before (`presence`, `pos_opp_mat`, `neg_opp_mat`, `species_order`, `countries_order`, `complete`, `permanently_failed`) plus new ones (see Step 10) |
| `gbif_presence_summary.csv` | Per location: species present (SInAS / GBIF / combined) |
| `gadm_level1_assignment.csv` | `gadm_sql` only: which GADM level-1 region feeds which location, and by which rule |
| `biotic_opportunity_qa.json` | QA report |

The previous `gbif_presence.pt` is kept as `gbif_presence_previous.pt`.

## Known limitation of `country_api` (inherited from the reused data)

GBIF reports occurrences per **country code**. Sub-national SInAS locations
that share a code with their country (Hawaii and Alaska with the United
States, the Canary Islands with Spain, …) all get that country's presence.
This over-marks presence on islands. Fixing it needs a new GBIF fetch with
sub-national (GADM) data: that is what `PRESENCE_SOURCE = "gadm_sql"`
does. The locations affected are listed in the QA report
(`shared_country_codes`).

## How to run

1. Install (once): `pip install pandas numpy scipy torch aiohttp
   nest_asyncio country_converter tqdm`
2. Run notebooks 01 and 03 first; keep the previous `gbif_presence.pt`
   in `../data/`.
3. *Run All*. Without new fetches this takes a few minutes; fetching new
   species takes roughly 1 minute per 5,000 species.

## Step 0 · Data sources and configuration

| Input | Source | How to obtain |
|---|---|---|
| `sinas_matched_species.csv`, `matched_globi_network*.csv`, `taxonomy_backbone.csv` | Notebook 03 | run notebook 03 |
| `complete_country_features_master.csv` | Notebook 01 (location list and ISO3 codes) | run notebook 01 |
| `gbif_presence.pt` (previous) | Previous GBIF fetch: GBIF occurrence search API, `facet=country` per taxon | already in `data/` (made by `fetch_gbif_presabs_country_pos_neg_split.ipynb`) |
| New GBIF counts | GBIF.org occurrence search API, <https://api.gbif.org/v1/occurrence/search>; GBIF data are licensed CC0 / CC BY / CC BY-NC per dataset | **automatic** (Step 4) |

When publishing results that use GBIF occurrence data, GBIF asks for a
citation of the data; for API counts like these, cite GBIF.org with the
access date (see the metadata file).

In [1]:
import asyncio
import base64
import csv
import getpass
import hashlib
import json
import os
import random
import time
import unicodedata
import urllib.error
import urllib.request
import warnings
import zipfile
from collections import Counter, defaultdict
from pathlib import Path
from dotenv import load_dotenv

import aiohttp
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
from tqdm.auto import tqdm

c:\Users\simon\Documents\GitHub\horizon-scanner\.venv\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


### Configuration

In [2]:
# ---------------------------------------------------------------- paths
load_dotenv(Path.cwd().parent / ".env")

DATA = Path.cwd().parent / "data"
SINAS_CSV = DATA / "sinas_matched_species.csv"
NET_SPECIES_CSV = DATA / "matched_globi_network.csv"
NET_HIGHER_CSV = DATA / "matched_globi_network_higher_order.csv"
TAXONOMY_CSV = DATA / "taxonomy_backbone.csv"
AUDIT_CSV = DATA / "name_matching_audit.csv"
MASTER_CSV = DATA / "complete_country_features_master.csv"
PREVIOUS_PRESENCE = [DATA / "gbif_presence.pt",
                     DATA / "gbif_presence_split.pt"]  # first found used
FACET_CACHE = DATA / "cache" / "gbif_country_facets.jsonl"
FACET_FAILED = DATA / "cache" / "gbif_country_facets_failed.jsonl"

# ---------------------------------------------------------------- outputs
PRESENCE_OUT = DATA / "gbif_presence.pt"
SUMMARY_OUT = DATA / "gbif_presence_summary.csv"
QA_OUT = DATA / "biotic_opportunity_qa.json"

# ---------------------------------------------------------------- GBIF fetch
FETCH_MISSING = True  # fetch species without previous GBIF data
GBIF_SEARCH_URL = "https://api.gbif.org/v1/occurrence/search"
CONTACT_EMAIL = os.getenv("GBIF_EMAIL")  # <- set this
MAX_CONCURRENT = 30  # parallel requests (old notebook used 50)
REQUEST_TIMEOUT = 30  # seconds per request
MAX_RETRIES = 4  # attempts per request within one pass
RETRY_PASSES = 3  # extra passes over the taxa that still failed
RETRY_WAIT_S = (30, 60, 120)  # wait before retry pass 1, 2, 3
FLUSH_EVERY = 100  # write the cache to disk every N answers
# Extra filters for NEW fetches. Empty = same query as the previous fetch
# (keeps the reused and new data comparable). A stricter, better query is
# e.g. {"occurrenceStatus": "PRESENT", "basisOfRecord": [
#   "HUMAN_OBSERVATION", "OBSERVATION", "MACHINE_OBSERVATION",
#   "PRESERVED_SPECIMEN", "MATERIAL_SAMPLE", "OCCURRENCE"]}
# (no absence, fossil or zoo / botanic-garden records), but then use it
# for a full re-fetch, not only for the new species.
GBIF_EXTRA_FILTERS = {}
MIN_RECORDS = 1  # GBIF records needed to call a species present

# ---------------------------------------------------------------- source
# Which GBIF presence to use (see the header for the full comparison):
#   "country_api": presence per GBIF country code. Reuses the previous
#        fetch; new species are fetched with one API request each.
#        Sub-national SInAS locations (Hawaii, Canary Islands, ...) get
#        their whole country's presence.
#   "gadm_sql":    GBIF SQL downloads (one job each; needs a GBIF account).
#        Countries split into several SInAS locations are resolved with
#        GADM level-1 regions, so Hawaii only gets Hawaiian records.
PRESENCE_SOURCE = "country_api"
# gadm_sql only: where the presence of all OTHER countries comes from:
#   "sql":      a fresh species x country SQL download (same filters for
#               every location; no per-species API requests at all)
#   "existing": the country_api data of Steps 3-4
GADM_COUNTRY_SOURCE = "sql"
# Record filter for the SQL downloads (absences, fossils and zoo /
# botanic-garden specimens excluded). GBIF SQL syntax.
SQL_WHERE = ("speciesKey IS NOT NULL AND occurrenceStatus = 'PRESENT' "
             "AND basisOfRecord <> 'FOSSIL_SPECIMEN' "
             "AND basisOfRecord <> 'LIVING_SPECIMEN'")
# Records in a split country without a GADM region (no or bad
# coordinates) go to that country's mainland location.
ASSIGN_UNLOCATED_TO_MAINLAND = True
# Reuse SQL downloads made earlier / on gbif.org: paste their keys here,
# e.g. {"country": "0012345-261001120000123", "level1": None}.
GBIF_SQL_KEYS = {"country": None, "level1": None}
GBIF_DOWNLOAD_URL = "https://api.gbif.org/v1/occurrence/download"
SQL_DIR = DATA / "cache" / "gbif_sql"
SQL_REGISTRY = SQL_DIR / "downloads.json"  # submitted jobs (reused)
SQL_POLL_S = 60  # seconds between status checks
SQL_MAX_WAIT_H = 12  # stop waiting after this (the job keeps running)
LEVEL1_OUT = DATA / "gadm_level1_assignment.csv"

USE_COUNTRY_API = (PRESENCE_SOURCE == "country_api"
                   or GADM_COUNTRY_SOURCE == "existing")
assert PRESENCE_SOURCE in ("country_api", "gadm_sql"), PRESENCE_SOURCE

# ---------------------------------------------------------------- partners
# Ranks of higher-taxon partners used for biotic opportunity. Order /
# class partners are present nearly everywhere and add little.
HOI_PARTNER_RANKS = ("GENUS", "FAMILY")
# A record about a genus is passed on to its member species. Adding
# "FAMILY" also passes on family-level records (broader, much larger).
HOI_INHERIT_RANKS = ("GENUS",)
# "uniform": every partner counts once (as before); "evidence": partners
# are weighted by log(1 + number of references).
PARTNER_WEIGHT = "uniform"

for f in (SINAS_CSV, NET_SPECIES_CSV, NET_HIGHER_CSV, TAXONOMY_CSV,
          MASTER_CSV):
    print(f"[{'OK ' if f.exists() else 'MISSING'}] {f}")
PREV_PATH = next((p for p in PREVIOUS_PRESENCE if p.exists()), None)
print(f"previous presence file: {PREV_PATH}")

[OK ] c:\Users\simon\Documents\GitHub\horizon-scanner\data\sinas_matched_species.csv
[OK ] c:\Users\simon\Documents\GitHub\horizon-scanner\data\matched_globi_network.csv
[OK ] c:\Users\simon\Documents\GitHub\horizon-scanner\data\matched_globi_network_higher_order.csv
[OK ] c:\Users\simon\Documents\GitHub\horizon-scanner\data\taxonomy_backbone.csv
[OK ] c:\Users\simon\Documents\GitHub\horizon-scanner\data\complete_country_features_master.csv
previous presence file: c:\Users\simon\Documents\GitHub\horizon-scanner\data\gbif_presence.pt


### Sub-national rules (used by `gadm_sql` only)

For every GBIF country code shared by several SInAS locations, each GADM
level-1 region of that country is assigned to locations as follows:

| `kind` | Meaning | Example |
|---|---|---|
| `exact` | the location **is** one or more level-1 regions (name match) | Hawaii ← "Hawaii"; Canary Islands ← "Canarias" |
| `containing` | the location is smaller than a level-1 region; it gets the region that contains it (an approximation, flagged) | Shetland ← "Scotland"; Vancouver Island ← "British Columbia" |
| `country` | no usable level-1 region; the location keeps the whole country's presence (as in `country_api`) | Kermadec Islands |

All remaining level-1 regions go to the **mainland** location (the one
location of that code without a rule, e.g. *United States*). Names are
matched case- and accent-insensitively as substrings.

A rule whose names match no region falls back to `country` with a warning.
The full assignment is written to `gadm_level1_assignment.csv`: **check it
once**, and add or correct rules here if needed.

In [3]:
LEVEL1_RULES = {
    # SInAS location: ([GADM level-1 name patterns], kind)
    "Hawaii": (["Hawaii"], "exact"),
    "Alaska": (["Alaska"], "exact"),
    "Canary Islands": (["Canarias", "Canary"], "exact"),
    "Balearic Islands": (["Baleares", "Balears", "Balearic"], "exact"),
    "Azores": (["Azores", "Acores"], "exact"),
    "Madeira": (["Madeira"], "exact"),
    "Sicily": (["Sicilia", "Sicily"], "exact"),
    "Sardinia": (["Sardegna", "Sardinia"], "exact"),
    "Corsica": (["Corse", "Corsica"], "exact"),
    "Tasmania": (["Tasmania"], "exact"),
    "Galapagos": (["Galapagos"], "exact"),
    "Crete": (["Crete", "Kriti"], "exact"),
    "Aegean": (["Aegean", "Aigaio"], "exact"),
    "Nicobar and Andaman Islands": (["Andaman"], "exact"),
    "Rodriguez Island": (["Rodrigues"], "exact"),
    "Zanzibar Island": (["Zanzibar"], "exact"),
    "Socotra Island": (["Socotra", "Suqutra"], "exact"),
    "Shetland Islands": (["Scotland"], "containing"),
    "Vancouver Island": (["British Columbia"], "containing"),
    "Lord Howe Islands": (["New South Wales"], "containing"),
    "Rapa Nui": (["Valparaiso"], "containing"),
    "Izu Islands": (["Tokyo"], "containing"),
    "Ogasawara Islands": (["Tokyo"], "containing"),
    "Fernando de Noronha": (["Pernambuco"], "containing"),
    "Kermadec Islands": ([], "country"),
    "Antipodes Island": ([], "country"),
}

### Helper functions

In [15]:
def run_async(coro):
    """Run a coroutine both in a plain script and inside Jupyter.

    Jupyter already runs an event loop, so ``asyncio.run`` would fail
    there; ``nest_asyncio`` lets us re-enter the running loop instead.
    """
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    import nest_asyncio
    nest_asyncio.apply()
    task = loop.create_task(coro)
    try:
        return loop.run_until_complete(task)
    except BaseException:  # interrupt or error: stop the workers too
        task.cancel()
        try:
            loop.run_until_complete(task)
        except BaseException:
            pass
        raise


def to_numpy(x):
    """Torch tensor or array-like -> numpy array."""
    return x.numpy() if hasattr(x, "numpy") else np.asarray(x)


def load_jsonl_cache(path):
    """Read a JSONL cache into {key: record}; half-written lines skipped."""
    cache = {}
    if path.exists():
        with open(path, encoding="utf-8") as fh:
            for line in fh:
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue
                cache[rec["_key"]] = rec
    return cache


class FetchStats:
    """Live counters shown on the progress bar and in the summary."""

    def __init__(self):
        self.ok = 0  # answered, counts stored
        self.no_data = 0  # permanent answer (e.g. HTTP 400/404) -> empty
        self.failed = 0  # gave up after MAX_RETRIES (retried next pass)
        self.rate_limited = 0  # HTTP 429 answers received
        self.retries = 0  # extra attempts after timeouts / 5xx / 429
        self.reasons = Counter()  # why requests failed

    def postfix(self):
        return {"ok": self.ok, "no_data": self.no_data,
                "failed": self.failed, "429": self.rate_limited,
                "retries": self.retries}


async def _pause_all(gate, seconds):
    """Pause every worker for ``seconds`` (shared back-off after a 429)."""
    if gate.is_set():
        gate.clear()
        await asyncio.sleep(seconds)
        gate.set()
    else:  # another worker is already pausing everyone
        await gate.wait()


async def _facet_request(session, taxon_key, gate, stats):
    """One taxon -> ("ok", counts) | ("no_data", reason) | ("failed", reason).

    * 200: counts per GBIF country code.
    * 429: GBIF asks us to slow down; ALL workers pause (Retry-After
      header if given, otherwise exponential back-off), then retry.
    * 5xx, timeout, connection error: wait and retry (up to MAX_RETRIES).
    * Other 4xx: the request itself is wrong (e.g. unknown key); stored as
      "no data" so it is not retried forever.
    """
    params = {"taxonKey": taxon_key, "limit": 0, "facet": "country",
              "facetLimit": 300, **GBIF_EXTRA_FILTERS}
    reason = "unknown"
    for attempt in range(MAX_RETRIES):
        if attempt:
            stats.retries += 1
        await gate.wait()
        try:
            async with session.get(
                GBIF_SEARCH_URL, params=params,
                timeout=aiohttp.ClientTimeout(total=REQUEST_TIMEOUT),
            ) as resp:
                if resp.status == 429:
                    stats.rate_limited += 1
                    reason = "http_429"
                    try:
                        wait = float(resp.headers.get("Retry-After", ""))
                    except ValueError:
                        wait = 10 * 2 ** attempt
                    await _pause_all(gate, wait + random.random())
                    continue
                if resp.status >= 500:
                    reason = f"http_{resp.status}"
                    await asyncio.sleep(2 ** attempt + random.random())
                    continue
                if resp.status != 200:
                    return "no_data", f"http_{resp.status}"
                data = await resp.json(content_type=None)
            facets = data.get("facets") or [{}]
            return "ok", {e["name"].upper(): int(e["count"])
                          for e in facets[0].get("counts", [])}
        except asyncio.TimeoutError:
            reason = "timeout"
        except aiohttp.ClientError as err:
            reason = type(err).__name__
        except (KeyError, TypeError, ValueError) as err:
            reason = f"bad_response_{type(err).__name__}"
        await asyncio.sleep(2 ** attempt + random.random())
    return "failed", reason


async def _fetch_pass(keys, cache_fh, fail_fh, desc):
    """Fetch ``keys`` with MAX_CONCURRENT workers, writing as results come.

    Every answer is appended to the cache file at once (flushed every
    FLUSH_EVERY answers and at the end), so an interruption loses at most
    the answers of the last few seconds.

    Returns
    -------
    (dict, dict, FetchStats)
        ({key: counts}, {key: failure reason}, stats)
    """
    queue = asyncio.Queue()
    for key in keys:
        queue.put_nowait(key)
    gate = asyncio.Event()
    gate.set()
    stats = FetchStats()
    results, failures = {}, {}
    pbar = tqdm(total=len(keys), desc=desc, unit="sp", dynamic_ncols=True)
    headers = {"User-Agent": f"IAS-horizon-scanner ({CONTACT_EMAIL})"}
    connector = aiohttp.TCPConnector(limit=MAX_CONCURRENT,
                                     ttl_dns_cache=300)

    async def worker(session):
        while True:
            if cache_fh.closed or fail_fh.closed:  # cell was stopped
                return
            try:
                key = queue.get_nowait()
            except asyncio.QueueEmpty:
                return
            status, payload = await _facet_request(session, key, gate,
                                                   stats)
            if status == "failed":
                stats.failed += 1
                stats.reasons[payload] += 1
                failures[key] = payload
                fail_fh.write(json.dumps({
                    "_key": str(key), "reason": payload,
                    "time": time.strftime("%Y-%m-%d %H:%M:%S")}) + "\n")
            else:
                counts = payload if status == "ok" else {}
                if status == "ok":
                    stats.ok += 1
                else:
                    stats.no_data += 1
                    stats.reasons[payload] += 1
                results[key] = counts
                rec = {"_key": str(key), "counts": counts}
                if status == "no_data":
                    rec["note"] = payload
                cache_fh.write(json.dumps(rec) + "\n")
            pbar.update(1)
            if pbar.n % FLUSH_EVERY == 0:
                cache_fh.flush()
                fail_fh.flush()
                pbar.set_postfix(stats.postfix(), refresh=False)

    try:
        async with aiohttp.ClientSession(headers=headers,
                                         connector=connector) as session:
            workers = [asyncio.create_task(worker(session))
                       for _ in range(MAX_CONCURRENT)]
            try:
                await asyncio.gather(*workers)
            finally:  # also on interrupt: stop workers cleanly
                for w in workers:
                    w.cancel()
                await asyncio.gather(*workers, return_exceptions=True)
    finally:
        cache_fh.flush()
        fail_fh.flush()
        pbar.set_postfix(stats.postfix())
        pbar.close()
    return results, failures, stats


def fetch_country_counts(taxon_keys):
    """GBIF country counts per taxon, cached and resumable.

    * Taxa already in FACET_CACHE are not fetched again.
    * Every answer is written to FACET_CACHE as soon as it arrives.
    * Failures are logged to FACET_FAILED (with the reason) and retried
      automatically in up to RETRY_PASSES extra passes, waiting
      RETRY_WAIT_S seconds before each pass.
    * Interrupting the cell is safe: re-run it and it continues where it
      stopped (failed taxa are simply not in the cache and get retried).

    Returns
    -------
    (dict, list)
        ({taxon_key: {iso2: count}}, [keys still failing])
    """
    FACET_CACHE.parent.mkdir(parents=True, exist_ok=True)
    cache = load_jsonl_cache(FACET_CACHE)
    keys = list(dict.fromkeys(int(k) for k in taxon_keys))
    todo = [k for k in keys if str(k) not in cache]
    print(f"   {len(keys):,} taxa | {len(keys) - len(todo):,} cached | "
          f"{len(todo):,} to fetch ({MAX_CONCURRENT} parallel)")
    reasons = Counter()
    with open(FACET_CACHE, "a", encoding="utf-8") as cache_fh, \
            open(FACET_FAILED, "a", encoding="utf-8") as fail_fh:
        for n_pass in range(RETRY_PASSES + 1):
            if not todo:
                break
            if n_pass:
                wait = RETRY_WAIT_S[min(n_pass, len(RETRY_WAIT_S)) - 1]
                print(f"   retry pass {n_pass}/{RETRY_PASSES}: "
                      f"{len(todo):,} taxa, waiting {wait} s first ...")
                time.sleep(wait)
            desc = "GBIF facets" if not n_pass else f"retry {n_pass}"
            results, failures, stats = run_async(
                _fetch_pass(todo, cache_fh, fail_fh, desc))
            for key, counts in results.items():
                cache[str(key)] = {"_key": str(key), "counts": counts}
            reasons.update(stats.reasons)
            print(f"   pass {n_pass}: {stats.postfix()}")
            todo = list(failures)
    if todo:
        warnings.warn(f"{len(todo):,} taxa still failing after "
                      f"{RETRY_PASSES} retry passes; re-run this cell "
                      f"later to retry them (see {FACET_FAILED.name})")
    if reasons:
        print("   reasons (all passes):", dict(reasons.most_common(6)))
    counts = {k: cache[str(k)]["counts"] for k in keys if str(k) in cache}
    return counts, todo


# ======================================================================
# GBIF SQL downloads (PRESENCE_SOURCE = "gadm_sql")
# ======================================================================
def _norm(text):
    """Lower-case, accent-free text for name matching."""
    text = unicodedata.normalize("NFKD", str(text))
    return text.encode("ascii", "ignore").decode().lower()


def _gbif_auth():
    """HTTP Basic auth header from GBIF_USER / GBIF_PWD (or a prompt).

    Set the environment variables before starting Jupyter to avoid the
    prompt; never write the password in the notebook.
    """
    user = os.environ.get("GBIF_USER") or input("GBIF username: ")
    pwd = os.environ.get("GBIF_PWD") or getpass.getpass("GBIF password: ")
    token = base64.b64encode(f"{user}:{pwd}".encode()).decode()
    return {"Authorization": f"Basic {token}"}


def _http(url, body=None, headers=None, timeout=120):
    """GET (or POST with ``body``) and return the response bytes."""
    req = urllib.request.Request(
        url, data=body, method="POST" if body is not None else "GET",
        headers={"User-Agent": f"IAS-horizon-scanner ({CONTACT_EMAIL})",
                 **(headers or {})})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.read()


def _load_registry():
    if SQL_REGISTRY.exists():
        return json.loads(SQL_REGISTRY.read_text())
    return {}


def submit_sql_download(label, sql):
    """Submit a GBIF SQL download, or reuse one made before.

    Reuse order: GBIF_SQL_KEYS[label] -> a job with the same SQL in
    SQL_REGISTRY -> submit a new job. Returns the download key.
    """
    SQL_DIR.mkdir(parents=True, exist_ok=True)
    if GBIF_SQL_KEYS.get(label):
        return GBIF_SQL_KEYS[label]
    registry = _load_registry()
    sql_hash = hashlib.md5(sql.encode()).hexdigest()
    if sql_hash in registry:
        key = registry[sql_hash]["key"]
        print(f"   {label}: reusing download {key}")
        return key
    body = json.dumps({
        "sendNotification": "@institution.org" not in CONTACT_EMAIL,
        "notificationAddresses": [CONTACT_EMAIL],
        "format": "SQL_TSV_ZIP", "sql": sql}).encode()
    try:
        key = _http(f"{GBIF_DOWNLOAD_URL}/request", body,
                    {**_gbif_auth(), "Content-Type": "application/json"}
                    ).decode().strip()
    except urllib.error.HTTPError as err:
        raise RuntimeError(f"GBIF refused the {label} SQL download "
                           f"(HTTP {err.code}): "
                           f"{err.read().decode(errors='ignore')[:800]}")
    registry[sql_hash] = {"key": key, "label": label, "sql": sql,
                          "submitted": time.strftime("%Y-%m-%d %H:%M")}
    SQL_REGISTRY.write_text(json.dumps(registry, indent=1))
    print(f"   {label}: submitted download {key}")
    return key


def wait_for_downloads(keys):
    """Poll GBIF until every download in {label: key} has SUCCEEDED.

    Prints one status line per check. Raises if a job fails; on timeout
    the jobs keep running at GBIF, so just re-run the cell later.
    """
    t0 = time.time()
    info = {}
    while True:
        line = []
        for label, key in keys.items():
            meta = json.loads(_http(f"{GBIF_DOWNLOAD_URL}/{key}"))
            info[label] = meta
            status = meta.get("status")
            line.append(f"{label}: {status}")
            if status in ("FAILED", "KILLED", "CANCELLED", "FILE_ERASED"):
                raise RuntimeError(f"GBIF download {key} ({label}) ended "
                                   f"with status {status}")
        print(f"   [{(time.time() - t0) / 60:5.1f} min] "
              + " | ".join(line), flush=True)
        if all(m.get("status") == "SUCCEEDED" for m in info.values()):
            return info
        if time.time() - t0 > SQL_MAX_WAIT_H * 3600:
            raise TimeoutError("Stopped waiting; the downloads keep running "
                               "at GBIF. Re-run this cell later.")
        time.sleep(SQL_POLL_S)


def fetch_download(key, meta):
    """Download the result zip once (resumes from scratch if incomplete)."""
    dest = SQL_DIR / f"{key}.zip"
    size = int(meta.get("size") or 0)
    if dest.exists() and (not size or dest.stat().st_size == size):
        return dest
    url = meta.get("downloadLink") or f"{GBIF_DOWNLOAD_URL}/request/" \
                                      f"{key}.zip"
    part = dest.with_suffix(".part")
    req = urllib.request.Request(
        url, headers={"User-Agent": f"IAS-horizon-scanner "
                                    f"({CONTACT_EMAIL})"})
    with urllib.request.urlopen(req, timeout=600) as resp, \
            open(part, "wb") as fh, \
            tqdm(total=size or None, unit="B", unit_scale=True,
                 desc=f"download {key}") as bar:
        while chunk := resp.read(1 << 20):
            fh.write(chunk)
            bar.update(len(chunk))
    part.replace(dest)
    return dest


def read_sql_result(zip_path, chunksize=5_000_000):
    """Yield the result table in chunks (lower-case column names).

    The table is the largest file in the zip; the last column holds the
    COUNT(*) value and is renamed ``n``.
    """
    with zipfile.ZipFile(zip_path) as zf:
        member = max(zf.infolist(), key=lambda i: i.file_size)
        with zf.open(member) as fh:
            for chunk in pd.read_csv(fh, sep="\t", dtype=str,
                                     chunksize=chunksize,
                                     quoting=csv.QUOTE_NONE,
                                     keep_default_na=False):
                chunk.columns = [c.lower() for c in chunk.columns]
                chunk = chunk.rename(columns={chunk.columns[-1]: "n"})
                chunk["n"] = pd.to_numeric(chunk["n"], errors="coerce")
                yield chunk


def assign_level1(units):
    """Assign GADM level-1 regions to SInAS locations (LEVEL1_RULES).

    Parameters
    ----------
    units : pandas.DataFrame
        Columns ``countrycode``, ``level1gid``, ``level1name``; one row per
        region, plus a row with ``level1gid == ""`` per country for
        records without a region.

    Returns
    -------
    pandas.DataFrame
        One row per (region, location) with the rule used.
    """
    rows, unmatched = [], []
    for code, grp in units.groupby("countrycode"):
        locs = [locations[c] for c in iso2_to_cols.get(code, [])]
        rules = {loc: LEVEL1_RULES[loc] for loc in locs
                 if loc in LEVEL1_RULES}
        mainland = [loc for loc in locs if loc not in rules]
        if len(mainland) > 1:
            warnings.warn(f"{code}: several mainland locations {mainland}")
        hit = Counter()
        for gid, name in zip(grp["level1gid"], grp["level1name"]):
            name_n = _norm(name) if gid else ""
            matches = {loc: kind for loc, (pats, kind) in rules.items()
                       if gid and kind != "country"
                       and any(_norm(p) in name_n for p in pats)}
            exact = [loc for loc, k in matches.items() if k == "exact"]
            hit.update(matches)
            targets = [(loc, "exact") for loc in exact]
            if not exact and (gid or ASSIGN_UNLOCATED_TO_MAINLAND):
                targets += [(loc, "mainland" if gid else "unlocated")
                            for loc in mainland]
            targets += [(loc, "containing") for loc, k in matches.items()
                        if k == "containing"]
            for loc, (_, kind) in rules.items():
                if kind == "country":
                    targets.append((loc, "country"))
            for loc, rule in targets:
                rows.append((code, gid, name, loc, rule))
        for loc, (pats, kind) in rules.items():
            if kind != "country" and not hit[loc]:
                unmatched.append(loc)
                rows += [(code, g, n, loc, "country_fallback")
                         for g, n in zip(grp["level1gid"],
                                         grp["level1name"])]
    if unmatched:
        warnings.warn(f"LEVEL1_RULES matched no GADM region for "
                      f"{unmatched}; they get the whole country's "
                      f"presence. Check gadm_level1_assignment.csv.")
    return pd.DataFrame(rows, columns=["countrycode", "level1gid",
                                       "level1name", "location", "rule"])


def row_normalised(rows, cols, weights, shape):
    """Sparse matrix with each row scaled to sum to 1 (empty rows stay 0)."""
    mat = sp.csr_matrix((weights, (rows, cols)), shape=shape)
    mat.sum_duplicates()
    sums = np.asarray(mat.sum(axis=1)).ravel()
    inv = np.divide(1.0, sums, out=np.zeros_like(sums), where=sums > 0)
    return sp.diags(inv) @ mat

## Step 1 · Species universe and locations

The species order **must match the model notebook**:
`sorted(SInAS species ∪ species-level GloBI endpoints)`, built from the
outputs of notebook 03. Locations are the 289 SInAS locations of the
country-feature master table (notebook 01), sorted by name.

In [5]:
print("Step 1: species universe and locations")
sinas = pd.read_csv(SINAS_CSV, low_memory=False)
net_sp = pd.read_csv(NET_SPECIES_CSV, low_memory=False)
net_ho = pd.read_csv(NET_HIGHER_CSV, low_memory=False)
taxonomy = pd.read_csv(TAXONOMY_CSV, low_memory=False)
master = pd.read_csv(MASTER_CSV, low_memory=False)

sinas["gbif_id"] = sinas["gbif_id"].astype("int64")
species_ids = sorted(
    set(sinas["gbif_id"])
    | set(net_sp["source_gbif_id"].dropna().astype("int64"))
    | set(net_sp["target_gbif_id"].dropna().astype("int64")))
sp_to_idx = {k: i for i, k in enumerate(species_ids)}

locations = sorted(master["location"].dropna().astype(str).str.strip()
                   .unique())
co_to_idx = {name: i for i, name in enumerate(locations)}
sinas_locs = set(sinas["location"].astype(str).str.strip())
missing_locs = sorted(sinas_locs - set(locations))
if missing_locs:
    warnings.warn(f"SInAS locations missing from the master table: "
                  f"{missing_locs}")
S, C = len(species_ids), len(locations)
print(f"   species: {S:,} | locations: {C}")

Step 1: species universe and locations
   species: 173,249 | locations: 289


## Step 2 · Locations → GBIF country codes

GBIF facets use ISO 3166-1 alpha-2 codes, including territories (PR, GU,
RE, …). The ISO3 code of each location comes from the master table and is
converted with `country_converter`. Locations that share one code (a
country and its sub-national units) all receive that code's GBIF presence;
they are listed in the QA report.

In [6]:
print("Step 2: location -> GBIF country code")
import country_converter as coco  # noqa: E402

iso3 = (master.drop_duplicates("location").set_index("location")["ISO3"]
        .reindex(locations))
need = iso3.isna()
if need.any():  # fall back on the location name
    iso3[need] = coco.convert(names=list(iso3.index[need]), to="ISO3",
                              not_found=None)
iso2 = pd.Series(coco.convert(names=list(iso3.fillna("")), src="ISO3",
                              to="ISO2", not_found=None), index=locations)
iso2 = iso2.where(iso2.astype(str).str.len() == 2)

iso2_to_cols = defaultdict(list)
for loc, code in iso2.dropna().items():
    iso2_to_cols[code.upper()].append(co_to_idx[loc])
shared_codes = {code: [locations[i] for i in cols]
                for code, cols in iso2_to_cols.items() if len(cols) > 1}
print(f"   codes: {len(iso2_to_cols)} | locations without a code: "
      f"{iso2.isna().sum()} | codes shared by >1 location: "
      f"{len(shared_codes)}")
pd.Series({k: ", ".join(v) for k, v in shared_codes.items()},
          name="locations").to_frame().head(15)

XAD not found in ISO3


Step 2: location -> GBIF country code


XCA not found in ISO3
XKO not found in ISO3
ZNC not found in ISO3
XPI not found in ISO3
XSP not found in ISO3


   codes: 249 | locations without a code: 6 | codes shared by >1 location: 21


,locations
GR,"Aegean, Crete, Greece"
US,"Alaska, Hawaii, United States of America"
TF,"Amsterdam Island, Crozet Islands, Kerguelen Is..."
NZ,"Antipodes Island, Kermadec Islands, New Zealand"
SH,"Ascension, Saint Helena, Tristan da Cunha"
AU,"Australia, Lord Howe Islands, Tasmania"
PT,"Azores, Madeira, Portugal"
ES,"Balearic Islands, Canary Islands, Spain"
BQ,"Bonaire, Saba, Sint Eustatius"
BR,"Brazil, Fernando de Noronha"


## Step 3 · Reuse the previous GBIF presence

The previous file stores presence per *old* species ID. Old IDs were the
key of the exact name matched, so a subspecies or synonym had its own ID;
notebook 03 now rolls these up to the species. Old IDs are therefore
mapped through the name-matching audit (`usageKey → species_key`) and the
rows of the same species are combined (present if any of them is).

The previous presence already contains the SInAS records of that time;
SInAS is added again in Step 5, so nothing is lost or double counted.

In `gadm_sql` mode with `GADM_COUNTRY_SOURCE = "sql"` the previous file is
still read (for the comparison in Step 9), but its presence is not used.

In [7]:
print("Step 3: reuse previous GBIF presence")
presence_gbif = np.zeros((S, C), dtype=bool)
has_gbif_data = np.zeros(S, dtype=bool)
prev_info = {"file": str(PREV_PATH) if PREV_PATH else None}

if PREV_PATH is not None:
    prev = torch.load(PREV_PATH, map_location="cpu", weights_only=False)
    old_presence = to_numpy(prev["presence"]).astype(bool)
    old_species = [int(k) for k in prev["species_order"]]
    old_countries = [str(c).strip() for c in prev["countries_order"]]
    col_map = np.array([co_to_idx.get(c, -1) for c in old_countries])

    usage_to_species = {}
    if AUDIT_CSV.exists():
        audit = pd.read_csv(AUDIT_CSV, low_memory=False,
                            usecols=["usageKey", "species_key"])
        audit = audit.dropna()
        usage_to_species = dict(zip(audit["usageKey"].astype("int64"),
                                    audit["species_key"].astype("int64")))
    how = {"direct": 0, "via_audit": 0, "unmapped": 0}
    rows_old, rows_new = [], []
    for i, key in enumerate(old_species):
        if key in sp_to_idx:
            new, kind = sp_to_idx[key], "direct"
        elif usage_to_species.get(key) in sp_to_idx:
            new, kind = sp_to_idx[usage_to_species[key]], "via_audit"
        else:
            how["unmapped"] += 1
            continue
        how[kind] += 1
        rows_old.append(i)
        rows_new.append(new)
    ok_cols = col_map >= 0
    if USE_COUNTRY_API:  # gadm_sql + "sql": only kept for the QA comparison
        for i_old, i_new in zip(rows_old, rows_new):
            presence_gbif[i_new, col_map[ok_cols]] |= \
                old_presence[i_old, ok_cols]
        failed_before = {int(k) for k in prev.get("permanently_failed",
                                                  [])}
        has_gbif_data[[n for o, n in zip(rows_old, rows_new)
                       if old_species[o] not in failed_before]] = True
    prev_info.update(how, old_species=len(old_species),
                     old_countries_not_found=[
                         c for c, j in zip(old_countries, col_map)
                         if j < 0],
                     previous_complete=bool(prev.get("complete", False)))
    print(f"   previous species: {len(old_species):,} -> {how}")
else:
    warnings.warn("No previous presence file found; everything will be "
                  "fetched in Step 4 (FETCH_MISSING) or left empty.")

Step 3: reuse previous GBIF presence
   previous species: 167,357 -> {'direct': 156030, 'via_audit': 9203, 'unmapped': 2124}


## Step 4 · Fetch species without previous GBIF data

Species of the universe that are not in the previous file (typically SInAS
species without GloBI interactions, now kept by notebook 03) are fetched
with the same query as before: occurrence counts per country for the
taxon (`facet=country`, one light request per species).

**Tracking and caching**

* The progress bar counts **species** and shows live counters: `ok`,
  `no_data` (GBIF answered that the request is invalid), `failed` (gave
  up for now), `429` (GBIF asked us to slow down) and `retries`.
* Every answer is appended to `data/cache/gbif_country_facets.jsonl` as
  it arrives (flushed every `FLUSH_EVERY` answers).
* Failures are logged with their reason in
  `data/cache/gbif_country_facets_failed.jsonl` and retried automatically
  in up to `RETRY_PASSES` extra passes.
* On HTTP 429 **all** workers pause together (GBIF's `Retry-After` if
  given), instead of every worker hammering on.
* Interrupting is safe: re-run the cell and it continues where it stopped.

Set `FETCH_MISSING = False` to skip the step. Skipped automatically in
`gadm_sql` mode with `GADM_COUNTRY_SOURCE = "sql"`.

In [16]:
print("Step 4: fetch species without previous GBIF data")
to_fetch = [species_ids[i] for i in np.where(~has_gbif_data)[0]]
failed = []
if USE_COUNTRY_API and FETCH_MISSING and to_fetch:
    counts, failed = fetch_country_counts(to_fetch)
    for key, by_code in counts.items():
        i = sp_to_idx[key]
        has_gbif_data[i] = True
        for code, n in by_code.items():
            if n >= MIN_RECORDS:
                presence_gbif[i, iso2_to_cols.get(code, [])] = True
    print(f"   fetched {len(counts):,} | failed {len(failed):,}")
elif not USE_COUNTRY_API:
    print("   skipped (gadm_sql mode: Step 4b downloads all species)")
else:
    print(f"   {len(to_fetch):,} species without GBIF data "
          f"({'skipped' if to_fetch else 'none'})")

Step 4: fetch species without previous GBIF data
   0 species without GBIF data (none)


## Step 4b · GADM approach: GBIF SQL downloads (`PRESENCE_SOURCE = "gadm_sql"`)

Skipped in `country_api` mode. Instead of one API request per species,
GBIF computes the counts for **all** species at once with its SQL
download service; you get one zip per query and a **DOI** to cite.

| Query | Content | Used for |
|---|---|---|
| `country` | `speciesKey × countryCode` counts (all countries) | every location with its own GBIF country code (if `GADM_COUNTRY_SOURCE = "sql"`) |
| `level1` | `speciesKey × countryCode × GADM level-1 region` counts, only for the country codes shared by several SInAS locations | splitting those countries (US, ES, PT, IT, …) |

Both use the record filter `SQL_WHERE` (present, no fossils, no zoo or
garden specimens).

**What you need**

* A free GBIF account (<https://www.gbif.org/user/profile>). Set the
  environment variables `GBIF_USER` and `GBIF_PWD` before starting
  Jupyter, or type them when asked. Never write the password in the
  notebook.
* Patience: GBIF runs the jobs on its cluster, typically within minutes
  to a few hours. The cell polls every `SQL_POLL_S` seconds. The status
  is also visible at <https://www.gbif.org/user/download>, and with
  `CONTACT_EMAIL` set you get an e-mail when a job is ready.

**Caching:** submitted jobs are stored in `cache/gbif_sql/downloads.json`
and the zips in `cache/gbif_sql/`. Re-running the cell (also after an
interruption or a timeout) reuses the same jobs and files, and never
submits twice. To reuse a download made elsewhere, paste its key in
`GBIF_SQL_KEYS`.

GBIF's SQL service is labelled *experimental*. If GBIF rejects the query
(HTTP 400), the error message is printed; usually a column name changed.
The list of columns is at
<https://api.gbif.org/v1/occurrence/download/describe/sql>.

In [12]:
print("Step 4b: GADM approach (GBIF SQL downloads)")
sql_info = {}
level1_table = None
if PRESENCE_SOURCE != "gadm_sql":
    print("   skipped (PRESENCE_SOURCE = 'country_api')")
else:
    split_codes = sorted(shared_codes)
    code_filter = " OR ".join(f"countryCode = '{c}'" for c in split_codes)
    queries = {"level1": (
        "SELECT speciesKey, countryCode, level1Gid, level1Name, COUNT(*) "
        f"AS n FROM occurrence WHERE {SQL_WHERE} AND ({code_filter}) "
        "GROUP BY speciesKey, countryCode, level1Gid, level1Name")}
    if GADM_COUNTRY_SOURCE == "sql":
        queries["country"] = (
            "SELECT speciesKey, countryCode, COUNT(*) AS n FROM occurrence "
            f"WHERE {SQL_WHERE} GROUP BY speciesKey, countryCode")
    keys = {label: submit_sql_download(label, sql)
            for label, sql in queries.items()}
    meta = wait_for_downloads(keys)
    zips = {label: fetch_download(keys[label], meta[label])
            for label in keys}
    sql_info = {label: {"key": keys[label], "doi": meta[label].get("doi"),
                        "created": meta[label].get("created"),
                        "total_records": meta[label].get("totalRecords"),
                        "sql": queries[label]} for label in keys}

    # --- country-level presence (all codes that are NOT split) ---------------
    split_cols = {c for code in split_codes for c in iso2_to_cols[code]}
    if GADM_COUNTRY_SOURCE == "sql":
        presence_gbif[:] = False
        has_gbif_data[:] = True  # absent from the table = 0 records
        for chunk in tqdm(read_sql_result(zips["country"]),
                          desc="country table", unit="chunk"):
            sp_i = pd.to_numeric(chunk["specieskey"], errors="coerce") \
                .map(sp_to_idx)
            ok = sp_i.notna() & (chunk["n"] >= MIN_RECORDS)
            for code, idx in zip(chunk.loc[ok, "countrycode"].str.upper(),
                                 sp_i[ok].astype(int)):
                cols = [c for c in iso2_to_cols.get(code, [])
                        if c not in split_cols]
                presence_gbif[idx, cols] = True
    else:  # keep Steps 3-4, only redo the split countries below
        presence_gbif[:, sorted(split_cols)] = False

    # --- split countries: species x level-1 region ---------------------------
    parts = []
    for chunk in tqdm(read_sql_result(zips["level1"]),
                      desc="level-1 table", unit="chunk"):
        sp_i = pd.to_numeric(chunk["specieskey"], errors="coerce") \
            .map(sp_to_idx)
        ok = sp_i.notna() & (chunk["n"] >= MIN_RECORDS)
        parts.append(pd.DataFrame({
            "sp": sp_i[ok].astype(int),
            "countrycode": chunk.loc[ok, "countrycode"].str.upper(),
            "level1gid": chunk.loc[ok, "level1gid"],
            "level1name": chunk.loc[ok, "level1name"]}))
    level1_table = pd.concat(parts, ignore_index=True)
    units = (level1_table[["countrycode", "level1gid", "level1name"]]
             .drop_duplicates(["countrycode", "level1gid"]))
    level1_map = assign_level1(units)
    level1_map.to_csv(LEVEL1_OUT, index=False)

    unit_key = list(zip(level1_map["countrycode"], level1_map["level1gid"]))
    unit_index = {u: i for i, u in enumerate(dict.fromkeys(unit_key))}
    obs_unit = pd.Series(list(zip(level1_table["countrycode"],
                                  level1_table["level1gid"]))).map(unit_index)
    kept = obs_unit.notna().to_numpy()  # regions assigned to no location
    assign = sp.csr_matrix(
        (np.ones(len(level1_map)),
         ([unit_index[u] for u in unit_key],
          level1_map["location"].map(co_to_idx).to_numpy())),
        shape=(len(unit_index), C))
    obs = sp.csr_matrix(
        (np.ones(int(kept.sum())),
         (level1_table["sp"].to_numpy()[kept],
          obs_unit[kept].astype(int).to_numpy())),
        shape=(S, len(unit_index)))
    presence_gbif |= (obs @ assign).toarray() > 0

    print(f"   level-1 regions: {len(units):,} in {len(split_codes)} split "
          f"countries | rules used: "
          f"{level1_map['rule'].value_counts().to_dict()}")
    print(f"   GBIF presences now: {presence_gbif.sum():,}")
    for label, inf in sql_info.items():
        print(f"   cite {label}: GBIF.org ({inf['created'][:10]}) GBIF "
              f"Occurrence Download https://doi.org/{inf['doi']}")

Step 4b: GADM approach (GBIF SQL downloads)
   skipped (PRESENCE_SOURCE = 'country_api')


## Step 5 · SInAS presence and the combined matrix

Every SInAS row (native, alien or other status) marks its species as
present in its location. **`presence` = GBIF ∪ SInAS.** It means
"recorded there", not "native there": GBIF includes alien populations.

In [13]:
print("Step 5: presence matrix")
presence_sinas = np.zeros((S, C), dtype=bool)
rows = sinas["gbif_id"].map(sp_to_idx)
cols = sinas["location"].astype(str).str.strip().map(co_to_idx)
ok = rows.notna() & cols.notna()
presence_sinas[rows[ok].astype(int), cols[ok].astype(int)] = True
presence = presence_gbif | presence_sinas
print(f"   presences: SInAS {presence_sinas.sum():,} | GBIF "
      f"{presence_gbif.sum():,} | combined {presence.sum():,}")

Step 5: presence matrix
   presences: SInAS 383,999 | GBIF 3,244,448 | combined 3,252,713


## Step 6 · Presence of higher taxa (HOI partners)

A genus (or family) partner counts as **present** in a location when at
least one species of that genus / family **in the species universe** is
present there. This is derived from the matrix above, so no extra
download is needed. It slightly underestimates presence for genera with
few species in the graph.

In [17]:
print("Step 6: presence of higher taxa")
RANK_COL = {"GENUS": "genus_id", "FAMILY": "family_id",
            "ORDER": "order_id", "CLASS": "class_id"}
tax_sp = taxonomy[taxonomy["rank"] == "SPECIES"].copy()
tax_sp = tax_sp[tax_sp["gbif_id"].isin(sp_to_idx)]
tax_sp["sp_idx"] = tax_sp["gbif_id"].map(sp_to_idx)

higher_ranks = sorted(set(HOI_PARTNER_RANKS) | set(HOI_INHERIT_RANKS))
members = {}  # (rank, taxon_key) -> array of species indices
for rank in higher_ranks:
    col = RANK_COL[rank]
    sub = tax_sp.dropna(subset=[col])
    for key, idx in sub.groupby(sub[col].astype("int64"))["sp_idx"]:
        members[(rank, int(key))] = idx.to_numpy()

partner_taxa = sorted(k for k in members if k[0] in HOI_PARTNER_RANKS)
ho_to_idx = {k: S + j for j, k in enumerate(partner_taxa)}
H = len(partner_taxa)
m_rows = np.concatenate([np.full(len(members[k]), j)
                         for j, k in enumerate(partner_taxa)]) \
    if H else np.array([], dtype=int)
m_cols = np.concatenate([members[k] for k in partner_taxa]) \
    if H else np.array([], dtype=int)
membership = sp.csr_matrix((np.ones(len(m_rows)), (m_rows, m_cols)),
                           shape=(H, S))
presence_higher = (membership @ presence.astype(np.float32)) > 0
print(f"   higher taxa: {H:,} "
      f"({pd.Series([k[0] for k in partner_taxa]).value_counts().to_dict()})")

Step 6: presence of higher taxa
   higher taxa: 45,347 ({'GENUS': 39871, 'FAMILY': 5476})


## Step 7 · Partner lists with valence

Each network edge gives each side a partner with a valence (from notebook
03): +1 → the partner goes into the **positive** list, −1 → into the
**negative** list, 0 → ignored (neutral or unknown interactions).

| Edge | Partner given to |
|---|---|
| species ↔ species | both species |
| species ↔ genus / family (`HOI_PARTNER_RANKS`) | the species gets the genus / family as partner |
| genus (`HOI_INHERIT_RANKS`) ↔ anything | every member species of that genus gets the other side as partner |

A partner that is the species itself, or a higher taxon containing the
species, is skipped. Each (species, partner, sign) is counted once
(`PARTNER_WEIGHT = "uniform"`), or weighted by the number of references
(`"evidence"`).

In [18]:
print("Step 7: partner lists")
edges = pd.concat([net_sp.assign(level="species"),
                   net_ho.assign(level="hoi")], ignore_index=True)
edges = edges.dropna(subset=["source_gbif_id", "target_gbif_id"])
edges["source_gbif_id"] = edges["source_gbif_id"].astype("int64")
edges["target_gbif_id"] = edges["target_gbif_id"].astype("int64")
weight_of = (np.log1p(edges["n_references"].fillna(1)).to_numpy()
             if PARTNER_WEIGHT == "evidence" else np.ones(len(edges)))

ancestors = defaultdict(set)  # species index -> its higher-taxon units
for rank in HOI_PARTNER_RANKS:
    col = RANK_COL[rank]
    for i, key in zip(tax_sp["sp_idx"], tax_sp[col]):
        if pd.notna(key) and (rank, int(key)) in ho_to_idx:
            ancestors[i].add(ho_to_idx[(rank, int(key))])


member_tab = pd.DataFrame(
    [(rank, key, int(i)) for (rank, key), idx in members.items()
     if rank in HOI_INHERIT_RANKS for i in idx],
    columns=["f_rank", "f_key", "focal"])
ancestor_tab = pd.DataFrame(
    [(f, p) for f, ps in ancestors.items() for p in ps],
    columns=["focal", "partner"])


def side_triplets(f_key, f_rank, valence, p_key, p_rank):
    """(focal, partner, sign, weight) rows for one side of every edge."""
    d = pd.DataFrame({"f_key": edges[f_key].to_numpy(),
                      "f_rank": edges[f_rank].to_numpy(),
                      "val": pd.to_numeric(edges[valence],
                                           errors="coerce").to_numpy(),
                      "p_key": edges[p_key].to_numpy(),
                      "p_rank": edges[p_rank].to_numpy(),
                      "weight": weight_of})
    d = d[d["val"].fillna(0) != 0]
    sp_partner = d["p_key"].map(sp_to_idx).where(d["p_rank"] == "SPECIES")
    ho_partner = pd.Series(list(zip(d["p_rank"], d["p_key"])),
                           index=d.index).map(ho_to_idx)
    d["partner"] = sp_partner.fillna(ho_partner)
    d = d.dropna(subset=["partner"])
    direct = d[d["f_rank"] == "SPECIES"].assign(
        focal=lambda x: x["f_key"].map(sp_to_idx)).dropna(subset=["focal"])
    inherited = d.merge(member_tab, on=["f_rank", "f_key"])
    out = pd.concat([direct, inherited], ignore_index=True)
    out["sign"] = np.sign(out["val"]).astype(int)
    return out[["focal", "partner", "sign", "weight"]].astype(
        {"focal": "int64", "partner": "int64"})


triplets = pd.concat([
    side_triplets("source_gbif_id", "source_rank", "source_valence",
                  "target_gbif_id", "target_rank"),
    side_triplets("target_gbif_id", "target_rank", "target_valence",
                  "source_gbif_id", "source_rank")], ignore_index=True)
triplets = triplets[triplets["focal"] != triplets["partner"]]
if len(ancestor_tab):  # drop "partner is my own genus / family"
    flag = triplets.merge(ancestor_tab, on=["focal", "partner"],
                          how="left", indicator=True)["_merge"]
    triplets = triplets[(flag == "left_only").to_numpy()]
triplets = (triplets.groupby(["focal", "partner", "sign"], as_index=False)
            ["weight"].max())
triplets["partner_is_species"] = triplets["partner"] < S
print(f"   triplets: {len(triplets):,} | species with positive partners: "
      f"{triplets.loc[triplets.sign > 0, 'focal'].nunique():,} | with "
      f"negative partners: "
      f"{triplets.loc[triplets.sign < 0, 'focal'].nunique():,}")

Step 7: partner lists
   triplets: 24,742,137 | species with positive partners: 147,753 | with negative partners: 94,306


## Step 8 · Biotic opportunity matrices

With **A** = species × partner weights (each row scaled to sum to 1) and
**P** = partner × location presence:

`pos_opp_mat = A_pos · P` and `neg_opp_mat = A_neg · P`,

i.e. the (weighted) fraction of a species' positive / negative partners
recorded in each location. One sparse matrix product replaces the old
per-species Python loop.

Species without any positive (negative) partner get 0, as before; the new
`n_pos_partners` / `n_neg_partners` arrays let the model tell "no known
partners" apart from "partners absent". The **species-level-only** version
(no HOI, as before) is also saved for comparison and ablation.

In [19]:
print("Step 8: biotic opportunity")
P_all = np.vstack([presence, presence_higher]).astype(np.float32)
U = P_all.shape[0]


def opportunity(trip_df, sign):
    """(S x C) fraction of partners with this sign present per location."""
    sub = trip_df[trip_df["sign"] == sign]
    A = row_normalised(sub["focal"].to_numpy(), sub["partner"].to_numpy(),
                       sub["weight"].to_numpy(), (S, U))
    return np.asarray(A @ P_all, dtype=np.float32)


pos_opp = opportunity(triplets, 1)
neg_opp = opportunity(triplets, -1)
sp_only = triplets[triplets["partner_is_species"]]
pos_opp_sp = opportunity(sp_only, 1)
neg_opp_sp = opportunity(sp_only, -1)
n_pos = np.bincount(triplets.loc[triplets.sign > 0, "focal"], minlength=S)
n_neg = np.bincount(triplets.loc[triplets.sign < 0, "focal"], minlength=S)
print(f"   mean opportunity (species with partners): "
      f"pos {pos_opp[n_pos > 0].mean():.3f} | "
      f"neg {neg_opp[n_neg > 0].mean():.3f}")

Step 8: biotic opportunity
   mean opportunity (species with partners): pos 0.315 | neg 0.213


## Step 9 · Quality assurance

* coverage: species with GBIF data, with any presence, with partners;
* how the previous file was re-keyed (direct / via audit / unmapped);
* the locations sharing a GBIF country code (known over-marking);
* comparison of the new and previous opportunity matrices for species
  present in both (correlation and mean absolute difference). Changes are
  expected from the new valences (generic associations are now neutral)
  and from HOI partners.

In [20]:
print("Step 9: QA")
qa = {
    "presence_source": PRESENCE_SOURCE,
    "gadm_country_source": (GADM_COUNTRY_SOURCE
                            if PRESENCE_SOURCE == "gadm_sql" else None),
    "gbif_sql_downloads": sql_info,
    "species": S, "locations": C, "higher_taxa": H,
    "species_with_gbif_data": int(has_gbif_data.sum()),
    "species_with_any_presence": int(presence.any(axis=1).sum()),
    "species_with_pos_partners": int((n_pos > 0).sum()),
    "species_with_neg_partners": int((n_neg > 0).sum()),
    "fetch_failed": [int(k) for k in failed],
    "previous_file": prev_info,
    "shared_country_codes": shared_codes,
    "locations_without_gbif_code": list(iso2.index[iso2.isna()]),
    "config": {"PRESENCE_SOURCE": PRESENCE_SOURCE,
               "SQL_WHERE": SQL_WHERE if PRESENCE_SOURCE == "gadm_sql"
               else None,
               "HOI_PARTNER_RANKS": HOI_PARTNER_RANKS,
               "HOI_INHERIT_RANKS": HOI_INHERIT_RANKS,
               "PARTNER_WEIGHT": PARTNER_WEIGHT,
               "MIN_RECORDS": MIN_RECORDS,
               "GBIF_EXTRA_FILTERS": GBIF_EXTRA_FILTERS},
}

if PREV_PATH is not None and "pos_opp_mat" in prev:
    old_pos = to_numpy(prev["pos_opp_mat"]).astype(np.float32)
    old_neg = to_numpy(prev["neg_opp_mat"]).astype(np.float32)
    old_idx = {k: i for i, k in enumerate(old_species)}
    both = [(old_idx[k], sp_to_idx[k]) for k in old_species
            if k in sp_to_idx]
    if both:
        o, n = map(np.array, zip(*both))
        o_cols = np.where(col_map >= 0)[0]
        n_cols = col_map[o_cols]
        cmp = {}
        for name, old_m, new_m in (("pos", old_pos, pos_opp),
                                   ("neg", old_neg, neg_opp)):
            a = old_m[np.ix_(o, o_cols)].ravel()
            b = new_m[np.ix_(n, n_cols)].ravel()
            r = (float(np.corrcoef(a, b)[0, 1])
                 if a.std() > 0 and b.std() > 0 else None)
            cmp[name] = {"pearson_r": r,
                         "mean_abs_diff": float(np.abs(a - b).mean()),
                         "mean_previous": float(a.mean()),
                         "mean_now": float(b.mean())}
        qa["comparison_with_previous"] = cmp
        print(pd.DataFrame(cmp).T.round(4).to_string())

summary = pd.DataFrame({
    "location": locations,
    "iso2": iso2.reindex(locations).to_numpy(),
    "species_present_sinas": presence_sinas.sum(axis=0),
    "species_present_gbif": presence_gbif.sum(axis=0),
    "species_present_total": presence.sum(axis=0),
    "shares_gbif_code": np.isin(
        locations, [loc for v in shared_codes.values() for loc in v]),
})
summary.sort_values("species_present_total", ascending=False).head(10)

Step 9: QA
     pearson_r  mean_abs_diff  mean_previous  mean_now
pos     0.6827         0.1673         0.2164     0.279
neg     0.4701         0.0962         0.0579     0.116


,location,iso2,species_present_sinas,species_present_gbif,species_present_total,shares_gbif_code
272,United States of America,US,11794,86797,86887,True
109,Hawaii,US,3271,86694,86719,True
3,Alaska,US,1002,86678,86679,True
200,Puerto Rico,PR,1760,81095,81109,False
281,Virgin Islands (U.S.),VI,540,80764,80773,False
271,United States Minor Outlying Islands,UM,270,80698,80701,False
88,France,FR,6380,43482,43726,True
60,Corsica,FR,2090,43117,43123,True
45,Canada,CA,3886,42985,43032,True
275,Vancouver Island,CA,91,42872,42872,True


## Step 10 · Write outputs

`gbif_presence.pt` keeps every key of the previous file, so the model
notebook loads it unchanged (point `gbif_presence_path` to this file).
New keys:

| Key | Content |
|---|---|
| `presence_sinas`, `presence_gbif` | the two sources of `presence` separately (bool, S × C) |
| `n_pos_partners`, `n_neg_partners` | number of positive / negative partners per species |
| `pos_opp_species_only`, `neg_opp_species_only` | opportunity without HOI partners (old definition, new valences) |
| `higher_taxa_order`, `presence_higher` | the genus / family partner units and their presence |
| `config`, `qa` | settings and QA of this run |

In [21]:
print("Step 10: writing outputs")
if PRESENCE_OUT.exists():
    PRESENCE_OUT.replace(PRESENCE_OUT.with_name("gbif_presence_previous.pt"))


def _t(a, dtype=None):
    """numpy -> torch tensor (optionally cast)."""
    a = np.ascontiguousarray(a)
    return torch.from_numpy(a.astype(dtype) if dtype else a)


torch.save({
    "presence": _t(presence),
    "pos_opp_mat": _t(pos_opp, np.float16),
    "neg_opp_mat": _t(neg_opp, np.float16),
    "species_order": species_ids,
    "countries_order": locations,
    "complete": len(failed) == 0 and bool(has_gbif_data.all()),
    "permanently_failed": [int(k) for k in failed],
    "presence_sinas": _t(presence_sinas),
    "presence_gbif": _t(presence_gbif),
    "n_pos_partners": _t(n_pos.astype(np.int32)),
    "n_neg_partners": _t(n_neg.astype(np.int32)),
    "pos_opp_species_only": _t(pos_opp_sp, np.float16),
    "neg_opp_species_only": _t(neg_opp_sp, np.float16),
    "higher_taxa_order": [list(k) for k in partner_taxa],
    "presence_higher": _t(presence_higher),
    "config": qa["config"],
    "qa": {k: v for k, v in qa.items() if k != "config"},
}, PRESENCE_OUT)
summary.to_csv(SUMMARY_OUT, index=False)
with open(QA_OUT, "w") as fh:
    json.dump(qa, fh, indent=2, default=str)
print(f"   {PRESENCE_OUT.name} | {SUMMARY_OUT.name} | {QA_OUT.name}\nDONE")

Step 10: writing outputs
   gbif_presence.pt | gbif_presence_summary.csv | biotic_opportunity_qa.json
DONE
